## Управление умной лампой жестами
## Smart lamp gestures manipulating
 
### Цель: реализовать механизм управления лампой через жесты и распознавание жестов
### Goal: to implement gesture control for a smart lightbulb and gesture recognition

### Суть алгоритма:
mediapipe расставляет точки на ладони, мы математически сравниваем их, чтобы распознать жест.

### Algorithm core:
mediapipe outputs hand points and we use them to determine gesture

### Первым делом импортируем библиотеки.
 
First things first let's import libraries.

In [ ]:
# надо установить модель
# we need mediapipe model so
#!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [26]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import numpy as np
import time
import uuid
import os

latest_result = None

In [22]:
def receive_result(result, output_image: mp.Image, timestamp_ms: int):
    global latest_result
    if result.hand_landmarks:
        print(f"[{timestamp_ms}ms] Обнаружено рук: {len(result.hand_landmarks)}")
        latest_result = result

In [23]:
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, 
                                       num_hands=2, 
                                       running_mode=mp.tasks.vision.RunningMode.LIVE_STREAM,
                                       min_hand_detection_confidence=0.5,
                                       min_hand_presence_confidence=0.5,
                                       min_tracking_confidence=0.5,
                                       result_callback=receive_result)
detector = vision.HandLandmarker.create_from_options(options)

I0000 00:00:1781041483.623180   11499 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781041483.626254   11511 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.5-arch1.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1781041483.647555   11504 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781041483.667372   11502 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [25]:
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # vibrant green

def draw_landmarks_on_image(rgb_image, detection_result):
  hand_landmarks_list = detection_result.hand_landmarks
  handedness_list = detection_result.handedness
  annotated_image = np.copy(rgb_image)

  # Loop through the detected hands to visualize.
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Draw the hand landmarks.
    mp_drawing.draw_landmarks(
      annotated_image,
      hand_landmarks,
      mp_hands.HAND_CONNECTIONS,
      mp_drawing_styles.get_default_hand_landmarks_style(),
      mp_drawing_styles.get_default_hand_connections_style())

    # Get the top left corner of the detected hand's bounding box.
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Draw handedness (left or right hand) on the image.
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

  return annotated_image

In [46]:
with python.vision.HandLandmarker.create_from_options(options) as detector:
    screen = cv2.VideoCapture(0)

    while screen.isOpened():
        success, frame = screen.read()
        if not success:
            print("Не удалось получить кадр с камеры.")
            continue
        frame = cv2.flip(frame, 1)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        
        timestamp_ms = int(time.time() * 1000)

        detector.detect_async(mp_image, timestamp_ms)
        
        annotated_frame = frame_rgb.copy() 
        
        if latest_result is not None:
            annotated_frame = draw_landmarks_on_image(annotated_frame, latest_result)
    
        final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
        cv2.imshow('MediaPipe Hand Landmarker', final_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    screen.release()
    cv2.destroyAllWindows()

I0000 00:00:1781042549.156947   15950 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781042549.159551   15962 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.5-arch1.1), renderer: Mesa Intel(R) UHD Graphics 620 (KBL GT2)
W0000 00:00:1781042549.167511   15954 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781042549.186433   15954 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[1781042551315ms] Обнаружено рук: 1
[1781042551451ms] Обнаружено рук: 1
[1781042551583ms] Обнаружено рук: 1
[1781042551715ms] Обнаружено рук: 2
[1781042551851ms] Обнаружено рук: 2
[1781042551983ms] Обнаружено рук: 2
[1781042552115ms] Обнаружено рук: 2
[1781042552251ms] Обнаружено рук: 2
[1781042552383ms] Обнаружено рук: 2
[1781042552515ms] Обнаружено рук: 2
[1781042552651ms] Обнаружено рук: 2
[1781042552783ms] Обнаружено рук: 2
[1781042552915ms] Обнаружено рук: 2
[1781042553051ms] Обнаружено рук: 2
[1781042553183ms] Обнаружено рук: 2
[1781042553315ms] Обнаружено рук: 2
[1781042553451ms] Обнаружено рук: 2
[1781042553583ms] Обнаружено рук: 2
[1781042553715ms] Обнаружено рук: 2
[1781042553851ms] Обнаружено рук: 2
[1781042553983ms] Обнаружено рук: 2
[1781042554115ms] Обнаружено рук: 2
[1781042554251ms] Обнаружено рук: 2
[1781042554383ms] Обнаружено рук: 2
[1781042554515ms] Обнаружено рук: 2
[1781042554651ms] Обнаружено рук: 2
[1781042554783ms] Обнаружено рук: 2
[1781042554915ms] Обнаружено

In [32]:
screen.release()
cv2.destroyAllWindows()